# Gaussian-linear task with SFMPE

Small end-to-end smoke run and SBC/TARP diagnostics against independent Gaussian-linear observations.

In [1]:
from pathlib import Path
import sys
import torch
import yaml

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT / "code" if (ROOT / "code" / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sfmpe.inference.SFMPE import SFMPE
from sfmpe.tasks.Gaussian import GaussianTask
from sfmpe.utils.metrics import check_sbc, check_tarp, run_sbc, run_tarp

with (ROOT / "configs" / "benchmark_gaussian.yaml").open(encoding="utf-8") as stream:
    task = GaussianTask(yaml.safe_load(stream))
pipeline = SFMPE(task, ROOT / "configs" / "sfmpe_benchmark_gaussian.yaml")
pipeline.run()

In [2]:
torch.manual_seed(31)
thetas, observations = task.simulate_dataset((10,))
num_posterior_samples = 64
sample_sets = []
for observation in observations:
    proposal = pipeline.manager.proposal
    proposal.params.x_0 = task.summarize(observation).unsqueeze(0)
    sample_sets.append(proposal.sample((num_posterior_samples,))[:, 0, :])
posterior_samples = torch.stack(sample_sets, dim=1)

ranks, dap_samples = run_sbc(
    thetas, posterior_samples, num_posterior_samples=num_posterior_samples
)
sbc_checks = check_sbc(
    ranks, task.sample_prior((len(thetas),)), dap_samples,
    num_posterior_samples=num_posterior_samples,
)
ecp, alpha = run_tarp(
    thetas, posterior_samples, num_bins=5
)
tarp_checks = check_tarp(ecp, alpha)
assert ranks.shape == (10, task.theta_dim)
assert posterior_samples.shape == (num_posterior_samples, 10, task.theta_dim)
assert torch.isfinite(ecp).all()
print("SBC KS p-values:", sbc_checks["ks_pvals"])
print("TARP (area-to-curve, KS p-value):", tarp_checks)

SBC KS p-values: tensor([0.7806, 0.7647])
TARP (area-to-curve, KS p-value): (0.009375002235174179, 0.9999999999999998)
